# 02 - Customer Analysis (RFM & Behavior)

**OmniStyle Customer Intelligence Platform**

This notebook digs into *why* customers behave the way they do: RFM
(Recency/Frequency/Monetary) patterns, loyalty tier behavior, and inactivity
trends. It answers the second level of business intelligence: **Why did it
happen?**

Prerequisite:

```bash
python -m src.feature_engineering
```


In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

from src.config import DATA_PROCESSED_DIR

pd.set_option("display.max_columns", 50)
plt.rcParams["figure.figsize"] = (10, 5)


In [ ]:
rfm = pd.read_csv(DATA_PROCESSED_DIR / "rfm_features.csv")
features = pd.read_csv(DATA_PROCESSED_DIR / "customer_features.csv")

print(rfm.shape, features.shape)
rfm.head()


## RFM distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
rfm["recency_days"].plot(kind="hist", bins=30, ax=axes[0], color="tomato")
axes[0].set_title("Recency (days since last purchase)")

rfm["frequency"].plot(kind="hist", bins=30, ax=axes[1], color="skyblue")
axes[1].set_title("Frequency (number of orders)")

rfm["monetary"].plot(kind="hist", bins=30, ax=axes[2], color="mediumseagreen")
axes[2].set_title("Monetary (total spend, $)")

plt.tight_layout()
plt.show()


## RFM score distribution

In [ ]:
rfm["RFM_score"].value_counts().sort_index().head(20).plot(kind="bar", color="purple", title="Top 20 RFM Score Combinations")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()


## Loyalty tier vs. spending behavior

In [ ]:
tier_summary = features.groupby("loyalty_tier").agg(
    customers=("customer_id", "count"),
    avg_monetary=("monetary", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_recency=("recency_days", "mean"),
).reindex(["Bronze", "Silver", "Gold", "Platinum"])

tier_summary


In [ ]:
tier_summary["avg_monetary"].plot(kind="bar", color="goldenrod", title="Average Lifetime Spend by Loyalty Tier")
plt.ylabel("Average Monetary Value ($)")
plt.tight_layout()
plt.show()


## Customer inactivity / retention analysis

In [ ]:
bins = [-1, 30, 90, 180, 365, 10_000]
labels = ["0-30 days", "31-90 days", "91-180 days", "181-365 days", "365+ days"]
features["inactivity_bucket"] = pd.cut(features["recency_days"], bins=bins, labels=labels)

inactivity_counts = features["inactivity_bucket"].value_counts().reindex(labels)
inactivity_counts.plot(kind="bar", color="indianred", title="Customer Inactivity Buckets")
plt.ylabel("Number of Customers")
plt.tight_layout()
plt.show()

inactivity_counts


## Tenure vs. purchase frequency

In [ ]:
plt.scatter(features["customer_tenure_days"], features["frequency"], alpha=0.2, s=10)
plt.xlabel("Customer Tenure (days)")
plt.ylabel("Purchase Frequency (orders)")
plt.title("Tenure vs. Purchase Frequency")
plt.tight_layout()
plt.show()


## Key takeaways

- Higher loyalty tiers (Gold/Platinum) show materially higher average
  monetary value, confirming the tier/spend relationship built into the
  synthetic data generator.
- A large share of the customer base falls into longer inactivity buckets
  (181+ days), which foreshadows the elevated churn rate we will see in
  `04_churn_prediction.ipynb`.
- There is a visible positive relationship between tenure and frequency for
  the more active segment of customers, though many long-tenured customers
  still show low frequency — these are prime candidates for the "At Risk"
  or "Hibernating" segments explored in `03_segmentation.ipynb`.
